# Fair Training-Free Experiments — E1-Centred
T1 diagnostics/bias → T2 DINO crops → T3 E1/E3/E4 ensemble → T4 TTA → T5 symptoms/disease prototypes. No additional gradient training. Read README for fairness/caching and input preparation. Defaults to audit-only.

## Setup
Use the suite environment kernel. On Colab, clone/install inside the remote runtime first. Select only the steps you want, smoke before full. Existing E1/E3/E4 weights are required unless the chosen step can reuse validated caches.

In [ ]:
from pathlib import Path
import sys, json, torch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common/dataset.py").is_file())
sys.path.insert(0, str(ROOT))
from approach3_training_free.run import Sequence
from approach3_training_free.methods import validate
CONFIG = json.loads((ROOT / "approach3_training_free/config.json").read_text())
STEPS = ["T1"]
MODE = "smoke"
RUN = False
ALLOW_DOWNLOAD = False
OUTPUT = None  # e.g. mounted Drive folder on Colab
CONFIG["steps"] = STEPS
validate(CONFIG)
print(torch.__version__, "CUDA:", torch.cuda.is_available())

## Data
Original7223 optimization-training images, original777 validation and official2000/1448 tests. All333 candidates are retained. The later1104-image holdout cannot tune E1/E3/E4 because they already trained on those images.

In [ ]:
from approach3_ood_backbones.data import shared_frames
from common.dataset import get_pairs
frames = shared_frames()
print({name: len(frame) for name, frame in frames.items()})
print("Candidates:", tuple(get_pairs().shape))

## Model
Frozen E1/E3/E4, original stored prompts and shared preprocessing. Formal clean scores must reproduce original records; DINO steps additionally preserve the original E1 robustness gate. DINO uses native processing, frozen CLS embeddings and training-only component prototypes.

In [ ]:
print("Checkpoint folder:", CONFIG["checkpoint_dir"])
print("Score caches:", CONFIG["score_caches"])
print("DINO cache:", CONFIG["dino_cache"])
print("No model is loaded in audit-only mode.")

## Training
No optimizer, backward pass or parameter updates. Prototype means use training labels only. Recipes are fixed before test scoring: bias0.5, CLIP/DINO0.8/0.2, equal ensemble weights, identity+horizontal flip, six sourced symptom descriptions with original-prompt fallback. Report all variants; do not choose coefficients or a final model from unseen-test scores.

In [ ]:
print("Selected:", STEPS, "Mode:", MODE, "RUN:", RUN)
print("Fixed bias:", CONFIG["bias"], "CLIP fraction:", CONFIG["clip_fraction"])
print("Equal ensemble:", CONFIG["ensemble_weights"], "TTA:", CONFIG["tta_views"])

## Evaluation
The corrected extended evaluator delegates the five core metrics to shared rules. Test labels are used only after predictions/recipes freeze. Smoke is two images per split, not full model accuracy. Input caches must match checkpoint SHA, exact image order,333 candidate rows and scope; unknown legacy caches are rejected.

In [ ]:
run_path = None
if RUN:
    sequence = Sequence(CONFIG, mode=MODE, allow_download=ALLOW_DOWNLOAD, output=OUTPUT)
    run_path = sequence.run()
else:
    print("Audit only; no weights loaded, no inference or experiment run.")

## Save
Each sequence writes a new unique folder. comparison.csv contains pair metrics; crop-only/disease-only accuracy is separate. All old checkpoints/results/splits remain unchanged. Label-free diagnostics and post-hoc error overlaps are descriptive only. Full GPU inference/reproduction remains to be performed.

In [ ]:
print("Results:", run_path if run_path else "No experiment outputs in audit-only mode")
if run_path:
    import pandas as pd
    display(pd.read_csv(run_path / "comparison.csv"))